In [89]:
import os
import warnings
warnings.filterwarnings("ignore")
from dotenv import load_dotenv
load_dotenv()

open_api_key = os.getenv("OPEN_API_KEY")
groq_api_key = os.getenv("GROQ_API_KEY")


In [90]:
from langchain_groq import ChatGroq 
from langchain_openai import ChatOpenAI
model = ChatGroq(model="openai/gpt-oss-120b",api_key=groq_api_key) 


In [91]:
text = 'How are you'

In [92]:
from langchain_core.messages import SystemMessage , HumanMessage

messages = [SystemMessage(content='Translate the text to Portugues'),
            HumanMessage(content=text)]


In [93]:
result = model.invoke(messages)
result


AIMessage(content='Como você está?', additional_kwargs={'reasoning_content': 'The user says "How are you". The developer instruction: "Translate the text to Portuguese". So we need to translate the text (the user message) to Portuguese. The user says "How are you". So we output "Como você está?" or "Como estás?" Portuguese. Probably "Como você está?" is standard. Provide translation.'}, response_metadata={'token_usage': {'completion_tokens': 83, 'prompt_tokens': 82, 'total_tokens': 165, 'completion_time': 0.177678992, 'completion_tokens_details': {'reasoning_tokens': 70}, 'prompt_time': 0.003529428, 'prompt_tokens_details': None, 'queue_time': 0.045396551, 'total_time': 0.18120842}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_a09bde29de', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--019d2fdf-ef7b-7c32-a379-1c8c56c1a000-0', usage_metadata={'input_tokens': 82, 'output_tokens': 83, 'total_tokens': 165})

In [94]:
from langchain_core.output_parsers import StrOutputParser
parser = StrOutputParser()


In [95]:
text = 'Amar is dumb'

In [96]:
from langchain_core.prompts import ChatPromptTemplate

def translator1(model,text):
    prompts = ChatPromptTemplate.from_messages([
        ("system","Translate the text to {language}"),("human", "{text}")])
    
    chain = prompts | model |StrOutputParser()
    
    return chain.invoke({"language":"Spanish",
                    "text" : text})
    
     

In [97]:
translator1(model,text)

'Amar es tonto.'

In [102]:
nums = [4,2,1,8]



In [104]:
print(sorted(nums))

[1, 2, 4, 8]


In [108]:
filename = 'companyPolicies.txt'
with open(filename,'r') as file:
    contents = file.read()
    print(contents)

1.	Code of Conduct

Our Code of Conduct outlines the fundamental principles and ethical standards that guide every member of our organization. We are committed to maintaining a workplace that is built on integrity, respect, and accountability.
Integrity: We hold ourselves to the highest ethical standards. This means acting honestly and transparently in all our interactions, whether with colleagues, clients, or the broader community. We respect and protect sensitive information, and we avoid conflicts of interest.
Respect: We embrace diversity and value each individual's contributions. Discrimination, harassment, or any form of disrespectful behavior is unacceptable. We create an inclusive environment where differences are celebrated and everyone is treated with dignity and courtesy.
Accountability: We take responsibility for our actions and decisions. We follow all relevant laws and regulations, and we strive to continuously improve our practices. We report any potential violations of 

In [110]:
from langchain_community.document_loaders import TextLoader
from langchain.text_splitter import CharacterTextSplitter

Splitting Documents into chunk

In [112]:
loader = TextLoader(filename)
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000,chunk_overlap = 0)
texts = text_splitter.split_documents(documents)
print(len(texts))

Created a chunk of size 1624, which is longer than the specified 1000
Created a chunk of size 1885, which is longer than the specified 1000
Created a chunk of size 1903, which is longer than the specified 1000
Created a chunk of size 1729, which is longer than the specified 1000
Created a chunk of size 1678, which is longer than the specified 1000
Created a chunk of size 2032, which is longer than the specified 1000
Created a chunk of size 1894, which is longer than the specified 1000


16


Embedding and storing

In [116]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
embeddings = HuggingFaceEmbeddings()
docsearch = Chroma.from_documents(texts, embeddings)
print('document ingested')

document ingested


Building LLM and retrieval pipeline

In [123]:
from langchain.chains import RetrievalQA
from langchain_groq import ChatGroq

model = ChatGroq(model="openai/gpt-oss-120b", api_key=groq_api_key)
qa = RetrievalQA.from_chain_type(
    llm=model,
    chain_type="stuff",  
    retriever=docsearch.as_retriever(),
    return_source_documents=False
)

In [125]:
query =  "What is mobile policy. Summarize it in bullet points"

qa.invoke(query)

{'query': 'What is mobile policy. Summarize it in bullet points',
 'result': '**Mobile Phone Policy – Key Points**\n\n- **Purpose:** Ensure mobile device use aligns with company values, legal requirements, and security best‑practice.  \n- **Acceptable Use:**  \n  - Primarily for work‑related tasks.  \n  - Limited personal use allowed if it doesn’t interfere with duties.  \n- **Security:**  \n  - Protect the device and login credentials.  \n  - Be cautious with apps, links, and downloads from unknown sources.  \n  - Report any security concerns or suspicious activity promptly.  \n- **Confidentiality:**  \n  - Do not send sensitive company information via unsecured messaging or email.  \n  - Discuss company matters discreetly, especially in public spaces.  \n- **Cost Management:**  \n  - Keep personal calls/texts separate from company accounts.  \n  - Reimburse the company for any personal charges on company‑issued phones.  \n- **Compliance:**  \n  - Follow all applicable laws and regula

Adding Prompts

In [152]:
prompt_template = """
You are an expert assistant.

Use ONLY the context below to answer the question.

Rules:
- Do not use outside knowledge
- If unsure, say "I don't know"
- Keep answers concise and clear

Context:
{context}

Question:
{question}

Answer:
"""

In [153]:
from langchain.prompts import PromptTemplate

PROMPT = PromptTemplate(
    template=prompt_template, input_variables=["context", "query"]
)

chain_type_kwargs = {"prompt": PROMPT}

In [154]:
qa = RetrievalQA.from_chain_type(llm=model, 
                                 chain_type="stuff", 
                                 retriever=docsearch.as_retriever(), 
                                 chain_type_kwargs=chain_type_kwargs, 
                                 return_source_documents=False)



In [155]:
query = "Can I eat in company vehicles?"
qa.invoke(query)

{'query': 'Can I eat in company vehicles?', 'result': "I don't know."}

Giving memory

In [159]:
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain
memory = ConversationBufferMemory(memory_key = "chat_history", return_message = True)

In [161]:
qa = ConversationalRetrievalChain.from_llm(llm=model, 
                                           chain_type="stuff", 
                                           retriever=docsearch.as_retriever(), 
                                           memory = memory, 
                                           get_chat_history=lambda h : h, 
                                           return_source_documents=False)

In [162]:
history = []

In [163]:
query = "What is mobile policy?"
result = qa.invoke({"question":query}, {"chat_history": history})
print(result["answer"])

The **Mobile Phone Policy** is the set of rules and expectations that govern how employees may use mobile devices (such as company‑issued smartphones or personal phones used for work) while they are on the job. Its purpose is to make sure that mobile‑phone use aligns with the organization’s values, legal obligations, and security best‑practice standards.

Key elements of the policy include:

| Area | What the policy requires |
|------|---------------------------|
| **Acceptable Use** | Mobile devices should be used mainly for work‑related tasks. Limited personal use is allowed as long as it does not interfere with job duties. |
| **Security** | Protect the device and login credentials, be careful when downloading apps or clicking unknown links, and promptly report any security concerns or suspicious activity. |
| **Confidentiality** | Do **not** send sensitive company information through unsecured messaging apps or email, and be discreet when discussing company matters in public places

In [164]:
history.append((query, result["answer"]))

In [165]:
query = "List points in it?"
result = qa({"question": query}, {"chat_history": history})
print(result["answer"])

**Key Points of the Mobile Phone Policy**

| Area | What the policy requires |
|------|---------------------------|
| **Purpose** | Ensure mobile‑phone use aligns with company values, legal requirements, and security best practices. |
| **Acceptable Use** | • Primarily for work‑related tasks.<br>• Limited personal use is allowed only if it does not interfere with job duties. |
| **Security** | • Protect the device and login credentials.<br>• Be cautious when downloading apps or clicking unknown links.<br>• Report any security concerns or suspicious activity promptly. |
| **Confidentiality** | • Do **not** send sensitive company information via unsecured messaging apps or email.<br>• Discuss company matters discreetly, especially in public spaces. |
| **Cost Management** | • Keep personal usage separate from company accounts.<br>• Reimburse the company for any personal charges on a company‑issued phone. |
| **Compliance** | • Follow all applicable laws and regulations (e.g., data‑protec

In [166]:
query = "What is the aim of it?"
result = qa({"question": query}, {"chat_history": history})
print(result["answer"])

The aim of the Mobile Phone Policy is to promote the **responsible and secure use of mobile devices** within the organization, ensuring that employees’ use of phones aligns with company values, legal compliance, and ethical standards. It seeks to protect company information, maintain security, manage costs, and uphold relevant laws and regulations.


In [167]:
def qa():
    memory = ConversationBufferMemory(memory_key = "chat_history", return_message = True)
    qa = ConversationalRetrievalChain.from_llm(llm=model, 
                                               chain_type="stuff", 
                                               retriever=docsearch.as_retriever(), 
                                               memory = memory, 
                                               get_chat_history=lambda h : h, 
                                               return_source_documents=False)
    history = []
    while True:
        query = input("Question: ")
        
        if query.lower() in ["quit","exit","bye"]:
            print("Answer: Goodbye!")
            break
            
        result = qa({"question": query}, {"chat_history": history})
        
        history.append((query, result["answer"]))
        
        print("Answer: ", result["answer"])

In [ ]:
qa()